# CANN架构与昇腾NPU原理
前文已明确，算子是AI模型的基础计算单元、Tensor是数据核心载体，而高效AI计算离不开算子优化与硬件适配的深度协同，传统CPU难以满足复杂模型的并行计算需求。  

昇腾 NPU 作为专用AI芯片提供专属算力支撑，CANN架构则承担起衔接上层AI框架与底层NPU硬件的“软件桥梁”角色，本章将聚焦以下核心内容，为Ascend C算子开发筑牢软硬件协同基础。
- CANN架构介绍
- 昇腾NPU原理

---

## 1. CANN架构介绍
CANN（Compute Architecture for Neural Networks）是华为针对AI场景推出的异构计算架构，对上支持多种AI框架，对下服务AI处理器与编程，发挥承上启下的关键作用，是提升昇腾AI处理器计算效率的关键平台。  

简单来说，CANN就像AI芯片与上层应用之间的“翻译官+调度员”，把开发者写的AI算法代码，转换成芯片能高效执行的指令，同时优化算力分配，最大化AI芯片的性能。

### 1.1 什么是异构计算架构
异构计算架构是“使能硬件异构并行计算的软件栈”，最简单的结构就是通用CPU和专用处理器的并行计算组合，其最大的好处是使能多元算力，化解算力瓶颈，从而实现算力最大化。

<img src="./images/heterogeneous_computing_architecture.png" alt="heterogeneous_computing_architecture"  width="350px" >

### 1.2 为什么要用异构计算架构
传统CPU以标量计算为核心，仅能逐元素执行单一运算。而专用硬件可直接完成向量、矩阵级的并行计算，以16×16矩阵乘这一AI场景的核心运算为例，不同计算架构的效率差距极为显著。

<img src="./images/matrix_multiplication_example.png" alt="matrix_multiplication_example"  width="700px" >

在CPU上执行该计算时，每个时钟周期仅能完成一次标量乘加运算，核心计算逻辑如下：
```
for (int i=0; i<16; i++) {
    for (int j=0; j<16; j++) {
        for (int k=0; k<16; k++) {
            // 乘、加操作各占1个时钟周期，单次共需2个cycle
            c[i][j] += a[i][k] * b[k][j];
        }
    }
}
```
**总耗时周期**：Cycle = 16×16×16×2 = 8192。

基于 Vector 矢量硬件单元的专用架构，每个时钟周期可完成一组行与列的矢量乘加运算，核心计算逻辑如下：
```
for (int i=0; i<16; i++) {
    for (int j=0; j<16; j++) {
        // 一行与一列的所有元素同时完成乘加运算
        c[i][j] = a[i][:] *+ b[:][j];
    }
}
```
**总耗时周期**：Cycle = 16×16 = 256。

如果是集成度更高的Cube矩阵乘硬件单元，单个时钟周期即可完成整份矩阵的乘加运算，核心计算逻辑如下：
```
// 两个16×16矩阵一次性完成并行乘加
c[:][:] = a[:][:] × b[:][:];
```
**总耗时周期**：Cycle = 1。

由此可见，面对 AI 场景的密集型计算需求，让专用硬件单元承接核心计算任务，而 CPU 专注于逻辑判断、指令下发等通用任务，可实现 “专人干专事” 的算力最优分配，大幅提升整体计算效率。

### 1.3 CANN架构介绍
CANN正是针对昇腾NPU的**异构计算架构**，它采用分层解耦的设计，上层组件体现CANN的内部能力，下层对接硬件原子能力。通过开放的接口帮助开发者快速调用底层算力，完成计算加速。
- 提供高性能算子及通信算法，帮助开发者进行大模型并行加速，释放芯片澎湃算力
- 提供多种算子开发方式，支持开发者进行高效开发与迁移
- 全面开源，给开发者提供丰富参考实践，让开发者具备自主创新能力

<img src="./images/cann_software_architecture.png" alt="cann_software_architecture"  width="700px" >

### 1.4 CANN的环境验证
CANN环境准备好后，可通过工具或者接口进行环境验证。

#### 1.4.1 使用工具验证
开发环境部署完成后，可通过npu-smi工具查询当前芯片信息。可执行以下命令验证驱动安装情况。

In [ ]:
!npu-smi info

若环境部署正常，则会打印每张卡的功耗、温度、AICore使用率、内存用量等信息，效果如下：
```
+-------------------------------------------------------------------------------------------+
| npu-smi 25.5.0                            Version: 25.5.0                                 |
+----------------------+---------------+----------------------------------------------------+
| NPU   Name           | Health        | Power(W)    Temp(C)           Hugepages-Usage(page)|
| Chip                 | Bus-Id        | AICore(%)   Memory-Usage(MB)  HBM-Usage(MB)        |
+======================+===============+====================================================+
| 0     xxx            | OK            | 70.5        42                1232 / 1232          |
| 0                    | 0000:C1:00.0  | 0           4252 / 15137      0    / 32768         |
+======================+===============+====================================================+
```

#### 1.4.2 使用接口验证
开发环境部署完成后，可正常调用acl及pyacl接口。执行以下代码调用API，查询芯片信息。

In [ ]:
import acl
print(acl.get_soc_name())

成功后会返回芯片版本字符串，如果环境部署存在问题，则通过该接口获取芯片版本会失败返回None。

---

## 2. 昇腾NPU原理
在异构计算架构中，专用处理器承担密集型计算任务，实现算力多元化调度与效能最大化。下文以Atlas A2训练系列产品为例，拆解昇腾设备的异构计算实现逻辑，清晰呈现NPU的硬件架构与计算机制。

### 2.1 Host和Device
典型应用场景中，将Atlas A2加速卡插入服务器（或个人PC）后，程序的整体逻辑控制、任务调度均在CPU侧执行；当触发大规模数据密集型计算任务时，CPU侧会将待计算数据传输至NPU侧内存，由NPU完成专用并行计算，最终将结果回传CPU侧，形成“控制-计算”的异构协同闭环。

<img src="./images/host_and_device.png" alt="host_and_device"  width="700px" >

- **Host（主机侧）**：即服务器CPU及配套内存所在侧，核心负责任务发起、逻辑判断、数据传输调度及结果汇总。

- **Device（设备侧）**：即昇腾加速卡的NPU及专属内存所在侧，核心负责承接密集型计算任务，通过专用硬件单元实现高效并行运算。

### 2.2 NPU内部细节
明确Host与Device的协同关系后，进一步拆解NPU内部单元组成，理解其专用计算能力的硬件支撑。

<img src="./images/npu_processor.png" alt="npu_processor"  width="700px" >

- **内存**：NPU专属内存，用于存储CPU传输的待计算数据、计算过程中的中间结果及最终输出数据，为高速运算提供数据缓存支撑。

- **AI Core**：昇腾NPU的核心计算单元，专为矩阵、向量、标量等密集型计算任务设计，是算子加速执行的核心载体。

- **AI CPU**：负责处理不适合在AI Core上执行的任务，如轻量级逻辑运算、非并行化处理的辅助算子，补充AI Core的计算场景覆盖。

- **控制CPU**：专注于NPU整体运行控制，协调内部各单元的工作时序，保障计算流程有序推进。

- **任务调度器**：基于任务类型与硬件资源状态，实现计算任务在AI Core、AI CPU间的高效分配与动态调度，最大化硬件利用率。

- **数字视觉预处理模块**：专用图像硬件处理单元，负责图像解码、编码、格式转换等预处理任务，减少AI Core的非核心计算负载。

### 2.3 AI Core内部细节
AI Core作为NPU的计算核心，绝大多数算子的加速执行均在此完成。其架构延续传统芯片“计算-存储-控制”的三大核心模块，通过专用化设计实现极致并行效能，下文逐一解析各模块功能。

<img src="./images/ai_core_architecture.png" alt="ai_core_architecture"  width="700px" >

#### 2.3.1 计算单元
AI Core内置三类专用计算单元，分别适配矩阵、向量、标量不同维度的计算需求，实现分工协作与并行提速。

<img src="./images/computing_unit.png" alt="computing_unit"  width="700px" >

1. **矩阵计算单元（Cube Unit）**   
    核心负责矩阵乘加运算，搭配累加器实现高效数据累加。硬件层面支持高精度并行计算：FP16精度下，单时钟周期可完成16×16与16×16矩阵乘（4096次乘加运算）；INT8精度下，单时钟周期可完成16×32与32×16矩阵乘（8192次乘加运算）。累加器可将当前矩阵乘结果与历史中间结果叠加，天然适配卷积运算中偏置（bias）添加等场景。

2. **向量计算单元（Vector Unit）**  
    专注于向量级运算，支持FP16、FP32、Int32、Int8等多数据类型，覆盖基本算术运算与定制化向量操作。运算效能表现为：单时钟周期可完成两组128长度FP16向量的加/乘运算，或64个FP32/Int32向量的加/乘运算，适配激活函数、数据归一化等向量密集型任务。

3. **标量计算单元（Scalar Unit）**  
    承担标量运算与AI Core整体控制职责，相当于微型CPU。核心功能包括：循环控制、分支判断、地址计算与参数配置（为Cube/Vector单元提供数据地址及运算参数），同时支持基础算术运算，保障各计算单元的协同有序运行。

#### 2.3.2 存储系统 
由片上存储单元与数据通路组成，通过分层存储设计减少外部总线访问频次，降低延迟、提升带宽，为高速计算提供数据支撑。

<img src="./images/storage_system.png" alt="storage_system"  width="700px" >

1. **存储转换引擎**  
    负责AI Core内部不同缓冲区的数据读写管理，同时支持多种数据格式转换操作，如Padding（填充）、Transpose（转置）、Img2Col（3D图像转2D矩阵）等预处理/后处理操作。此外，可通过总线接口直接访问AI Core外部的低层级缓存，拓展数据访问范围。

2. **缓冲区**  
    包含L1缓冲区、L0A/L0B缓冲区、L0C缓冲区、统一缓冲区及标量缓冲区，核心作用是缓存高频复用数据与中间结果：一方面，将频繁访问的数据暂存片上，避免反复从外部读取，减少总线拥堵与功耗消耗；另一方面，存储神经网络各层计算的中间结果，为下一层运算快速提供数据，相较总线访问大幅降低延迟、提升运算效率。

3. **寄存器**  
    主要为标量计算单元服务，用于暂存标量数据、运算指令及控制参数，保障标量运算的高速执行。

#### 2.3.3 控制单元
作为AI Core的“指挥中枢”，负责全流程指令控制与时序协调，确保各单元并行运算的有序性与数据一致性。核心组成及功能如下：

<img src="./images/control_unit.png" alt="control_unit"  width="700px" >

- **系统控制模块**：管控任务块（AI Core最小计算任务粒度）的执行进程，任务块完成后执行中断处理与状态上报；若运算过程中出现错误，及时向任务调度器反馈错误状态。

- **指令缓存**：提前预取后续待执行指令，一次性读取多条指令缓存，避免指令逐条读取的延迟，提升指令执行效率。

- **标量指令处理队列**：指令解码后导入该队列，完成地址解码与运算控制，覆盖矩阵、向量、存储转换等各类指令。

- **指令发射模块**：读取标量指令处理队列中的指令地址与参数，解码后按指令类型分发至对应执行队列，标量指令则留存于该队列中执行。

- **指令执行队列**：分为矩阵运算队列、向量运算队列、存储转换队列，不同类型指令按顺序在对应队列中执行，实现并行流水线运算。

- **事件同步模块**：实时监控各指令流水线的执行状态，分析不同流水线的依赖关系，解决数据依赖与时序同步问题（如矩阵乘完成后再执行向量加法），保障运算结果正确性。

#### 2.3.4 总结
结合上述架构，昇腾NPU的计算加速流程可概括为：开发者通过CANN API编写的计算逻辑，经架构转换为一条条硬件可执行指令，下发至NPU后，由控制单元按类型分发至对应指令队列；AI Core内的矩阵、向量、存储转换等单元从队列中并行取指执行；针对存在先后依赖的任务（如矩阵乘后紧跟向量加法），开发者调用同步API，由事件同步模块管控流水线时序，确保依赖关系满足。

需说明的是，本文介绍的为AI Core基础架构，不同代际的昇腾NPU（如310、910B、910C）在硬件排布、数据通路设计上可能存在差异，具体特性将在后续内容中针对性讲解。

---

## 课后练习
本节总体介绍了CANN异构计算架构与昇腾NPU的硬件细节，请根据学习内容完成以下题目进行自测。

1. （判断题）传统CPU以标量计算为核心，而专用硬件可完成向量级并行计算。

2. （判断题）AI Core的指令执行队列实现了并行流水线运算，不同类型指令被分发到对应专属队列，队列内指令按序执行，队列间指令并行调度，最终实现整体的流水线运算。

3. （单选题）在昇腾NPU中，Host侧的核心职责是什么？  
    A. 执行密集型计算  
    B. 任务发起、逻辑判断、数据传输调度  
    C. 专用并行运算  
    D. 图像预处理  

4. （单选题）AI Core的计算单元中，负责矩阵乘加运算的是？  
    A. Vector Unit  
    B. Scalar Unit  
    C. Cube Unit  
    D. 存储转换引擎  

5. （多选题）CANN的优势包括哪些？（可多选）  
    A. 提供高性能算子及通信算法  
    B. 支持多种算子开发方式  
    C. 全面开源  
    D. 仅支持单一AI框架  

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/01.03_answer.txt